In [1]:
# initiate open router model
%pip install -U "langchain-openrouter"
%pip install -U "langchain"


import os
import getpass
from langchain_openrouter import ChatOpenRouter

# Set API key using secure input
api_key = getpass.getpass("Enter your OPENROUTER_API_KEY: ")
os.environ["OPENROUTER_API_KEY"] = api_key
print(f"API key set: {'*' * len(api_key)}")


model = ChatOpenRouter(model="auto")



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.2/337.2 kB 7.4 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 159.5 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 142.8 kB/s eta 0:00:00a 0:00:01
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.8
    Uninstalling langgraph-prebuilt-1.0.8:
      Successfully uninstalled langgraph-prebuilt-1.0.8
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.1.4
    Uninstalling langgraph-1.1.4:
      Successfully uninstalled langgraph-1.1.4
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.14
    Uninstalling langchain-1.2.14:
      Successfully uninstalled langchain-1.2.14
API key set: *************************************************************************


In [2]:
def direct_prompt(task):
    return f"""
Solve the following problem and return only the final answer.

Problem:
{task}
"""

In [3]:
def cot_prompt(task):
    return f"""
Solve the following problem step by step.

Show your reasoning clearly, then give the final answer.

Problem:
{task}
"""

In [4]:
tasks = [
    {
        "type": "math",
        "question": "If a train travels 60 km in 1.5 hours, what is its average speed?",
        "answer": "40 km/h"
    },
    {
        "type": "logic",
        "question": "All cats are mammals. Some mammals are black. Are all cats black?",
        "answer": "No"
    },
    {
        "type": "word_problem",
        "question": "John has twice as many apples as Mary. Together they have 18 apples. How many does Mary have?",
        "answer": "6"
    },
    {
        "type": "code",
        "question": "What is the bug?\n\nfor i in range(5):\n    print(i)\n    i += 1",
        "answer": "Modifying loop variable has no effect"
    },
    {
        "type": "algorithm",
        "question": "What is the time complexity of binary search?",
        "answer": "O(log n)"
    }
]

In [17]:
def run_experiment(tasks):
    results = []

    for task in tasks:
        # Direct
        direct_res = model.invoke(direct_prompt(task["question"]))

        # CoT
        cot_res = model.invoke(cot_prompt(task["question"]))

        results.append({
            "type": task["type"],
            "question": task["question"],
            "expected": task["answer"],
            "direct": direct_res,
            "cot": cot_res
        })

    return results

In [19]:
def evaluate(results):
    direct_correct = 0
    cot_correct = 0

    for r in results:
        if r["expected"].lower() in r["direct"].content.lower():
            direct_correct += 1
        if r["expected"].lower() in r["cot"].content.lower():
            cot_correct += 1

    return {
        "direct_accuracy": direct_correct / len(results),
        "cot_accuracy": cot_correct / len(results)
    }

In [20]:
results = run_experiment(tasks)
metrics = evaluate(results)

print(metrics)

{'direct_accuracy': 0.6, 'cot_accuracy': 0.4}


In [21]:
results

[{'type': 'math',
  'question': 'If a train travels 60 km in 1.5 hours, what is its average speed?',
  'expected': '40 km/h',
  'direct': AIMessage(content='40 km/h', additional_kwargs={'reasoning_content': 'We just need to compute average speed = distance/time = 60 km / 1.5 h = 40 km/h. Return final answer only.', 'reasoning_details': [{'type': 'reasoning.text', 'text': 'We just need to compute average speed = distance/time = 60 km / 1.5 h = 40 km/h. Return final answer only.', 'format': 'unknown', 'index': 0.0}]}, response_metadata={'model_name': 'openai/gpt-oss-120b', 'id': 'gen-1776154501-OmFFyvlSO1gHGThVqmdF', 'created': 1776154502, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter'}, id='lc_run--019d8b0e-e2cb-7591-af91-0741a7d65ce5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 103, 'output_tokens': 44, 'total_tokens': 147, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_de